# DBLP Outlier Repair v3.1

运行全部单元可先完成离线自测。要处理真实主表，将下一格的 RUN_FULL 改为 True。结果写入 Drive 的独立运行目录。

In [ ]:
RUN_FULL = False  # @param {type:"boolean"}
REFRESH_REVIEWS = False  # @param {type:"boolean"}
import base64, hashlib, io, json, subprocess, sys, zipfile
from pathlib import Path
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
if IN_COLAB and RUN_FULL:
    from google.colab import drive
    drive.mount('/content/drive')


## 解包程序并运行测试

In [ ]:
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='d6edd5edc59aca97da39190af42d32cef2cb20fe984b28cc967b61578d1a22fa'
ENGINE=(Path('/content') if IN_COLAB else Path.cwd())/'DBLP_v31_d6edd5edc59a'
ENGINE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    archive.extractall(ENGINE)
from importlib.util import find_spec
if any(find_spec(m) is None for m in ('pandas','numpy','lxml')):
    subprocess.run([sys.executable,'-m','pip','install','-r',str(ENGINE/'requirements.txt')],check=True)
def execute(args):
    with subprocess.Popen([sys.executable,'-u',*args],cwd=ENGINE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
        if process.wait():raise RuntimeError('运行失败，见上方日志')
execute(['-m','unittest','discover','-s','tests','-v'])


## 配置数据和缓存
主表可以填写 CSV 或原 data_quality_check.zip 的路径。Colab 未找到默认文件时会弹出上传框。

In [ ]:
cfg=json.loads((ENGINE/('config.json' if IN_COLAB else 'config.local.example.json')).read_text())
for key in ['input_csv','recovery_root','xml_path','dtd_path','cache_root','output_root','local_work_dir','legacy_cache_root']:
    cfg[key]=str(Path(cfg[key]).resolve())
cfg['candidate_csv']=str(ENGINE/'existing_candidate_ids.csv')
cfg['reviewed_rules']=str(ENGINE/'reviewed_record_exclusions.json')
# 自定义路径写在这里，例如 cfg['input_csv']='/content/drive/MyDrive/my_project/papers_features.csv'
# cfg['crossref_mailto']=''
print(json.dumps(cfg,ensure_ascii=False,indent=2))

def prepare_master(cfg, uploaded_file=None):
    import shutil, tempfile
    import pandas as pd
    source = Path(uploaded_file or cfg['input_csv']).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f'未找到主表 {source}。请在配置格选择第二轮主表 CSV 或 data_quality_check.zip。')
    if source.suffix.lower()=='.zip':
        with zipfile.ZipFile(source) as archive:
            matches=[i for i in archive.infolist() if Path(i.filename).name=='papers_features.csv' and '__MACOSX' not in Path(i.filename).parts]
            if len(matches)!=1:
                raise ValueError(f'ZIP 中有 {len(matches)} 份 papers_features.csv，请直接选择要使用的 CSV。')
            dest=Path(tempfile.mkdtemp(prefix='dblp_input_'))/'papers_features.csv'
            with archive.open(matches[0]) as src, dest.open('wb') as dst:
                shutil.copyfileobj(src,dst)
            source=dest
    header=set(pd.read_csv(source,nrows=0).columns)
    required={'paper_id','title_v1','submission_year','authors_v1_json','author_count_v1'} | {
        f'dblp_team_{stat}_{metric}_pre_t0' for stat in ('mean','max') for metric in ('publication_count','academic_age')}
    if required-header:
        raise ValueError('所选文件缺少主表字段：'+', '.join(sorted(required-header)))
    ids=pd.read_csv(source,usecols=['paper_id'],dtype=str,keep_default_na=False).paper_id
    if ids.eq('').any() or ids.duplicated().any():raise ValueError('主表 paper_id 有空值或重复值。')
    if cfg.get('candidate_csv'):
        candidates=set(pd.read_csv(cfg['candidate_csv'],usecols=['paper_id'],dtype=str).paper_id)
        missing=candidates-set(ids)
        if missing:raise ValueError(f'这份主表缺少 {len(missing)} 个候选 ID，请核对主表版本。')
    cfg['input_csv']=str(source.resolve())
    print(f'主表已就绪：{source}，共 {len(ids):,} 篇。')

if RUN_FULL:
    uploaded_file=None
    if IN_COLAB and not Path(cfg['input_csv']).is_file():
        from google.colab import files
        print(f"当前路径找不到主表：{cfg['input_csv']}。请选择 papers_features.csv 或原 data_quality_check.zip。")
        uploaded=files.upload()
        if len(uploaded)!=1:raise ValueError('请选择一个文件，或修改配置格的路径后重新运行。')
        # Colab may rename an uploaded file when a file with the same name already exists.
        import tempfile
        filename,content=next(iter(uploaded.items()))
        uploaded_file=Path(tempfile.mkdtemp(prefix='dblp_upload_'))/Path(filename).name
        uploaded_file.write_bytes(content)
        del uploaded,content
    prepare_master(cfg,uploaded_file)


## 运行全量候选核查
联网预算仅控制外部请求。程序在整个候选队列上执行离线筛查。阶段 ETA 与实时请求数会显示在本格。

In [ ]:
latest=None
if RUN_FULL:
    config=ENGINE/'colab.run.json'
    config.write_text(json.dumps(cfg,ensure_ascii=False,indent=2))
    result_root=Path(cfg['output_root'])
    before=set(result_root.glob('run_*'))
    cmd=['main.py','--config',str(config)]
    cmd.append('--audit-only')
    if REFRESH_REVIEWS:cmd.append('--refresh-reviews')
    execute(cmd)
    completed=[p for p in set(result_root.glob('run_*'))-before if (p/'summary.json').is_file()]
    if len(completed)!=1:raise RuntimeError('请从日志确认本次的结果目录')
    latest=completed[0]
else:
    print('离线自测已完成。需要处理实际主表时，在第一格启用 RUN_FULL。')


## 查看覆盖率、证据和修复前后对比

In [ ]:
if latest:
    import pandas as pd
    summary=json.loads((latest/'summary.json').read_text())
    assert summary['all_outlier_papers_visited'] and summary['papers_with_verdict']==summary['outlier_papers']
    print('结果目录',latest)
    for key in ('outlier_papers','verdict_counts','changed_papers','changed_cells','external_http_requests'):
        print(key,summary.get(key,0))
    for file in ('runtime_estimate.json','cache_reuse_report.json','progress.json'):
        if (latest/file).exists():print(file,(latest/file).read_text())
    print(pd.read_csv(latest/'changes.csv').head(20).to_string(index=False))

    assert summary['mode']=='AUDIT_ONLY' and summary['changed_cells']==0
    print('作品与版本明细',summary.get('distinct_occurrence_works'),summary.get('evidence_record_rows'))
    if (latest/'comparison_v3.json').exists():print((latest/'comparison_v3.json').read_text())
